# TIES4600 Machine Learning
## Ryhmätyö
Jäsenet: Joona Launonen, Anni Tarvainen, Santeri Leinonen

Dataset: Student Performance Factors

Linkki: https://www.kaggle.com/datasets/lainguyn123/student-performance-factors

Tavoite: Rakentaa koneoppimis malli, joka syötetyn datan perusteella tunnistaa testi tuloksiin eniten vaikuttavat tekijät ja osaa ennustaa opiskelijan testi tulokset.

Malli: LinearRegression

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import optuna

from sklearn.preprocessing import OneHotEncoder,StandardScaler,PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split,KFold, cross_val_score,cross_validate,cross_val_predict
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error,r2_score,root_mean_squared_error
from sklearn.linear_model import LinearRegression,Ridge,Lasso

In [ ]:
def load_data():
    # Read dataset from CSV file into a pandas DataFrame
    df = pd.read_csv("StudentPerformanceFactors.csv")

    # Drop all rows with any missing values
    df.dropna(inplace=True)

    # Optional: filter out rows with Exam_Score >= 78
    df = df[df["Exam_Score"] <= 100]

    # Transform exam scores using natural logarithm (target variable)
    exam_score = np.log(df["Exam_Score"])

    # Features: remove the target column from the dataset
    students = df.drop(columns=["Exam_Score"], axis=1)

    # Identify names of numeric feature columns
    numeric_features = students.select_dtypes(include=["number"]).columns.tolist()

    # Identify names of categorical feature columns
    categorical_columns = students.select_dtypes(include=["object"]).columns.tolist()

    # Return feature matrix, target, and lists of column names by type
    return students, exam_score, categorical_columns, numeric_features


# Load data and metadata for modeling
X, y, cat_cols, num_cols = load_data()

In [ ]:
def test_fitting(X_train, y_train, X_val, y_val, model):
    # Predict target values for training and validation sets
    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)

    # Compute R² scores for training and validation performance
    train_score = r2_score(y_train, y_train_pred)
    val_score = r2_score(y_val, y_val_pred)

    # Difference between training and validation R² (generalization gap)
    score_gap = train_score - val_score

    # Return diagnosis and detailed R² metrics
    return train_score, val_score, score_gap


In [ ]:
def build_model(categorical_columns, numeric_features, random_state=None):
    # Define preprocessing for categorical and numeric columns
    preprocessor = ColumnTransformer(
        transformers=[
            # One-hot encode categorical features, ignore unseen categories at prediction time
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
            # Standardize numeric features (zero mean, unit variance)
            ("num", StandardScaler(), numeric_features),
        ]
    )

    # Set up a linear regression model
    regressor = LinearRegression(
        n_jobs=-1,          # Use all available CPU cores
        fit_intercept=True, # Fit the intercept term
        copy_X=True,        # Copy input data to avoid modifying it in-place
        positive=False,     # Allow coefficients to take negative values
    )

    # Combine preprocessing and regression into a single pipeline
    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),  # First apply transformations
            ("regressor", regressor),        # Then fit/predict with linear regression
        ]
    )

    # Return the complete modeling pipeline
    return model

In [ ]:
# Define a K-Fold splitter (not actually used for splitting below, only for loop count)
kf = KFold(n_splits=100, shuffle=True, random_state=42)

# Lists to store metrics across runs
train_l = []
val_l = []
score_l = []

# Placeholder list for feature importance dataframes (currently unused)
importance_dfs = []

# Load features/target and column type metadata
X, y, cat_cols, num_cols = load_data()

# Build model using categorical and numerical column info
model = build_model(cat_cols, num_cols)

# Repeat random train/validation splits 100 times
# (Note: train_idx, val_idx from KFold are not used in the actual split)
for train_idx, val_idx in kf.split(X):
    # Random seed to vary the train/validation split each iteration
    seed = np.random.randint(0, 10000)

    # Create a random train/validation split
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.3, random_state=seed
    )

    # Fit model on the current training split
    model_fit = model.fit(X_train, y_train)

    # Evaluate model and compute train score, validation score, and their gap
    train_score, val_score, score_gap = test_fitting(
        X_train, y_train, X_val, y_val, model_fit
    )

    # Store metrics for this run
    train_l.append(train_score)
    val_l.append(val_score)
    score_l.append(score_gap)

# Report average metrics across all runs
print(f'val score {np.mean(val_l)}')
print(f'train score {np.mean(train_l)}')
print(f'gap score {np.mean(score_l)}')

# Baseline cross-validation score using sklearn's cross_val_score
print(f'cross val score: {np.mean(cross_val_score(model, X, y))} ')


In [ ]:
def plot_feature_importances(feature, conf, color, title, xlabel, ylabel):
    # Get the chosen colormap by name
    cmap = plt.colormaps[color]
    # Create a range of colors from the colormap for each feature
    colors = cmap(np.linspace(0.2, 0.85, len(feature)))

    # Create a new figure and axes with a fixed size
    fig, ax = plt.subplots(figsize=(10, 8))

    # Plot horizontal bar chart of feature importances
    ax.barh(feature, conf, color=colors)

    # Set plot title and axis labels
    ax.set(
        title=title,
        xlabel=xlabel,
        ylabel=ylabel,
    )

    # Adjust layout to prevent clipping of labels
    plt.tight_layout()
    return fig, ax  # Return figure and axes for further customization or saving


In [ ]:
# Create a DataFrame with feature names and their corresponding coefficients
importance_df = pd.DataFrame({
    "feature": model.named_steps["preprocessor"].get_feature_names_out(),
    "coefficient": model.named_steps["regressor"].coef_,
    "abs_coef": np.abs(model.named_steps["regressor"].coef_),  # magnitude of coefficients
})

# Sort features by absolute coefficient (importance) in ascending order
sorted_importance = importance_df.sort_values(by="abs_coef", ascending=True)

# Plot signed coefficients (shows direction: positive vs negative effect)
plot_feature_importances(
    feature=sorted_importance["feature"], 
    conf=sorted_importance["coefficient"],
    color="Reds", title="Feature Importances", 
    xlabel="Coefficient", ylabel="Features")

# Plot absolute coefficients (importance regardless of direction)
plot_feature_importances(
    feature=sorted_importance["feature"], 
    conf=sorted_importance["abs_coef"],
    color="inferno", title="Feature Importances", 
    xlabel="Absolute Coefficient", ylabel="Features")

# Render the plots
plt.show()
